# RSNA Hemorrhage Sequence Classification — Colab Training

Clones the [rsna-hemorrhage-cv](https://github.com/ptuan21/rsna-hemorrhage-cv) repo, downloads the dataset from Kaggle ([samali012/rsna-multiwindow-sequence-384-v3](https://www.kaggle.com/datasets/samali012/rsna-multiwindow-sequence-384-v3)), and trains on the Colab GPU.

**Before running:** Runtime -> Change runtime type -> select a GPU (T4 is fine).

In [ ]:
!git clone https://github.com/ptuan21/rsna-hemorrhage-cv.git
%cd rsna-hemorrhage-cv

In [ ]:
!pip install -q -r requirements.txt kaggle

## Kaggle credentials

Get your API token from https://www.kaggle.com/settings -> "Create New Token" (downloads `kaggle.json`), then run the cell below and upload that file when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select your kaggle.json here

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d samali012/rsna-multiwindow-sequence-384-v3 -p rsna_data --unzip

In [ ]:
# Sanity check: directory layout and split sizes should match the dataset README.
!ls rsna_data
!wc -l rsna_data/train.csv rsna_data/validation.csv rsna_data/test.csv

## Train

`--device auto` picks CUDA automatically on a Colab GPU runtime. The pipeline now includes:

- **Train-time augmentation** (flip, small rotation, brightness/contrast jitter) — sequence-consistent, disabled on val/test.
- **Bidirectional GRU** context over slice embeddings before attention pooling, so slice order/adjacency is no longer ignored.
- **Focal Loss** (`--focal-gamma`, default 2.0) + a **class-balanced sampler** to oversample rare classes like epidural.
- **Cosine LR annealing** over `--epochs`, **gradient clipping** (`--grad-clip-norm`, default 5.0), and **early stopping** (`--patience`, default 7 epochs with no val macro-F1 gain) — so it's safe to set `--epochs` generously.

If you hit `CUDA out of memory` on a smaller GPU, lower `--batch-size` and/or `--max-slices` (default 16).

In [ ]:
!python -m src.train --data-root rsna_data --checkpoint-dir checkpoints --epochs 40 --batch-size 8 --patience 7 --device auto

## Evaluate on the held-out test split

In [ ]:
!python -m src.evaluate --data-root rsna_data --split test --checkpoint checkpoints/best_model.pt --device auto --output-csv test_predictions.csv

## Save the trained checkpoint back to Google Drive (optional)

Colab runtimes are ephemeral — mount Drive and copy the checkpoint out before the session recycles.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/rsna-hemorrhage-cv
!cp checkpoints/best_model.pt test_predictions.csv /content/drive/MyDrive/rsna-hemorrhage-cv/